# Rossmann Demo - Run All

Notebook này chạy offline trong VS Code và gọi lại luồng Python mà backend đang sử dụng:

- `make_predictions(...)`: `backend/app/routers/predictions.py`
- `infer(...)` / `predict_frame(...)`: `backend/app/model/predict_model.py`
- `make_features(...)`: `backend/app/model/features.py`
- Dữ liệu đã xử lý: `data/processed/` và `data/merged/`
- Mô hình: `models/rossmann_sales_lgbm.joblib`

**Quy tắc bắt buộc:** nếu `Open = 0` thì kết quả là `0.00 EUR`, bất kể mô hình dự đoán gốc là bao nhiêu.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "backend" / "app" / "main.py").exists()), None)
assert ROOT is not None, "Không tìm thấy thư mục gốc chứa backend/app/main.py"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))

import numpy as np
import pandas as pd

print(f"Project root: {ROOT}")

Project root: D:\RossmannSalesForecast_FullProject\RossmannSalesForecast


## 1. Dữ liệu và mô hình đã lưu

Các dòng sau nạp đúng dữ liệu dự án và mô hình đã lưu, không huấn luyện lại.

In [2]:
from app.model.predict_model import infer, load_model, predict_frame
from app.model.features import make_features
from app.routers.predictions import PredictRequest, make_predictions, to_frame
from app.services.data_service import (
    get_stores,
    get_train,
    get_test,
    get_weather,
)

artifact = load_model()
model = artifact["model"]
stores = get_stores()
train = get_train()
test = get_test()

print("Đã nạp mô hình:", ROOT / "models" / "rossmann_sales_lgbm.joblib")
print("Số cửa hàng:", len(stores))
print("Train:", len(train), "dòng; Test:", len(test), "dòng")

Đã nạp mô hình: D:\RossmannSalesForecast_FullProject\RossmannSalesForecast\models\rossmann_sales_lgbm.joblib
Số cửa hàng: 1115
Train: 1017209 dòng; Test: 41088 dòng


## 2. Chỉnh kịch bản trong một ô

Thay đổi các biến dưới đây rồi chạy lại ô dự đoán. `StateHoliday` chỉ nhận `0`, `a`, `b`, hoặc `c`; `Promo`, `Open`, `SchoolHoliday` chỉ nhận `0/1`.

In [3]:
# ============ CHỈNH KỊCH BẢN TẠI Ô NÀY ============
Store = 1
Date = "2015-08-03"
Promo = 1
Open = 1
SchoolHoliday = 0
StateHoliday = "0"
# ==================================================

valid_stores = set(stores["Store"])
assert Store in valid_stores, f"Store phải thuộc {sorted(valid_stores)[0]}..{sorted(valid_stores)[-1]}"
assert Promo in (0, 1), "Promo chỉ nhận 0 hoặc 1"
assert Open in (0, 1), "Open chỉ nhận 0 hoặc 1"
assert SchoolHoliday in (0, 1), "SchoolHoliday chỉ nhận 0 hoặc 1"
assert StateHoliday in ("0", "a", "b", "c"), "StateHoliday phải là 0, a, b hoặc c"

request = PredictRequest(
    store=Store,
    date=pd.Timestamp(Date).date(),
    promo=Promo,
    open=Open,
    school_holiday=SchoolHoliday,
    state_holiday=StateHoliday,
)

In [4]:
# Gọi đúng hàm backend đang phục vụ web
backend_result = make_predictions([request])[0]

print("Kịch bản:", {
    "Store": Store,
    "Date": Date,
    "Promo": Promo,
    "Open": Open,
    "SchoolHoliday": SchoolHoliday,
    "StateHoliday": StateHoliday,
})
print(f"Doanh thu dự đoán: {backend_result['predicted_sales']:.2f} EUR")
print("Kết quả đóng cửa:", "0.00 EUR" if Open == 0 else "Không áp dụng quy tắc đóng cửa")
print("Có dữ liệu actual trên cùng điều kiện:", backend_result["conditions_match_actual"])
if backend_result["actual_sales"] is not None:
    print(f"Doanh thu thực: {backend_result['actual_sales']:.2f} EUR")
print("Ngoài phạm vi dữ liệu lịch sử/test:", backend_result["is_out_of_range"])

Kịch bản: {'Store': 1, 'Date': '2015-08-03', 'Promo': 1, 'Open': 1, 'SchoolHoliday': 0, 'StateHoliday': '0'}
Doanh thu dự đoán: 5967.62 EUR
Kết quả đóng cửa: Không áp dụng quy tắc đóng cửa
Có dữ liệu actual trên cùng điều kiện: False
Ngoài phạm vi dữ liệu lịch sử/test: False


## 3. Đối chiếu hàm backend

Ô này chạy `infer(...)` độc lập trên khung dữ liệu đầu vào của backend. Với `Open = 1`, kết quả phải trùng số dự đoán của hàm backend; với `Open = 0`, hàm backend buộc kết quả bằng `0 EUR`.

In [5]:
# Khôi phục cùng khung dữ liệu mà router truyền cho infer
timestamp = pd.Timestamp(request.date)
if pd.Timestamp("2013-01-01") <= timestamp <= pd.Timestamp("2015-07-31"):
    row = train.loc[train["Store"].eq(request.store) & train["Date"].eq(timestamp)]
    if not row.empty:
        direct_frame = row.iloc[[0]].copy()
        direct_frame.loc[:, "Promo"] = request.promo
        direct_frame.loc[:, "Open"] = request.open
        direct_frame.loc[:, "SchoolHoliday"] = request.school_holiday
        direct_frame.loc[:, "StateHoliday"] = request.state_holiday
    else:
        direct_frame = to_frame([request])
elif pd.Timestamp("2015-08-01") <= timestamp <= pd.Timestamp("2015-09-17"):
    row = test.loc[test["Store"].eq(request.store) & test["Date"].eq(timestamp)]
    if not row.empty:
        direct_frame = row.iloc[[0]].copy()
        direct_frame.loc[:, "Promo"] = request.promo
        direct_frame.loc[:, "Open"] = request.open
        direct_frame.loc[:, "SchoolHoliday"] = request.school_holiday
        direct_frame.loc[:, "StateHoliday"] = request.state_holiday
    else:
        direct_frame = to_frame([request])
else:
    direct_frame = to_frame([request])

direct_raw = infer(direct_frame)
direct_backend_value = 0.0 if request.open == 0 else round(float(direct_raw[0]), 2)
web_value = float(backend_result["predicted_sales"])
delta = abs(direct_backend_value - web_value)

print(f"infer(...) trước quy tắc làm tròn: {float(direct_raw[0]):.6f} EUR")
print(f"Giá trị đối chiếu sau quy tắc Open: {direct_backend_value:.2f} EUR")
print(f"make_predictions(...): {web_value:.2f} EUR")
print(f"Sai khác tuyệt đối: {delta:.2f} EUR")

assert delta <= 0.01, f"Kết quả lệch: {delta:.6f} EUR"
if request.open == 0:
    assert web_value == 0.0

# Kiểm tra bắt buộc quy tắc cửa hàng đóng với cùng kịch bản
closed_request = request.model_copy(update={"open": 0})
closed_result = make_predictions([closed_request])[0]
print(f"Thử Closed: Open=0 -> {closed_result['predicted_sales']:.2f} EUR")
assert closed_result["predicted_sales"] == 0.0

infer(...) trước quy tắc làm tròn: 5967.618813 EUR
Giá trị đối chiếu sau quy tắc Open: 5967.62 EUR
make_predictions(...): 5967.62 EUR
Sai khác tuyệt đối: 0.00 EUR
Thử Closed: Open=0 -> 0.00 EUR


## 4. MAE, RMSE, RMSPE thực tế

Các metric dưới đây lấy từ `results/evaluation_metrics.json`, được tạo bởi script đánh giá trên **holdout thời gian 42 ngày**, mô hình validation chưa huấn luyện lại trên tập holdout.

In [6]:
import json

metrics_path = ROOT / "results" / "evaluation_metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
validation = metrics["validation"]

raw = get_train()
cutoff = pd.Timestamp(metrics["validation_dates"][0])
valid_mask = raw["Date"].ge(cutoff)

metric_table = pd.DataFrame([{
    "MAE (EUR)": validation["mae"],
    "RMSE (EUR)": validation["rmse"],
    "RMSPE": validation["rmspe"],
    "RMSPE (%)": validation["rmspe"] * 100,
    "WAPE": validation["wape"],
}], index=["Model validation"])

display(metric_table)
print("Nguồn metric:", metrics_path)
print("Tập train:", metrics["train_dates"][0], "đến", metrics["train_dates"][1])
print("Tập validation:", metrics["validation_dates"][0], "đến", metrics["validation_dates"][1])
print("Số dòng validation:", metrics["validation_rows"])
print("Kiểm tra lại số dòng trong dữ liệu train:", int(valid_mask.sum()))

if "metrics" in artifact:
    saved_metrics = artifact["metrics"]["validation"]
    print("Metric trong file model:", saved_metrics)
    assert saved_metrics == validation

,MAE (EUR),RMSE (EUR),RMSPE,RMSPE (%),WAPE
Model validation,509.82,811.59,0.12134,12.134,0.08493


Nguồn metric: D:\RossmannSalesForecast_FullProject\RossmannSalesForecast\results\evaluation_metrics.json
Tập train: 2013-01-01 đến 2015-06-19
Tập validation: 2015-06-20 đến 2015-07-31
Số dòng validation: 46830
Kiểm tra lại số dòng trong dữ liệu train: 46830
Metric trong file model: {'mae': 509.82, 'rmse': 811.59, 'rmspe': 0.12134, 'wape': 0.08493}
